# Funkcje okienkowe w SQL (T-SQL) — kompletne notatki referencyjne

Przykłady na modelu: `dim_Klienci` (ID_Klienta, Nazwa), `fact_Sprzedaz` (ID_Klienta, ID_Placowki, DataSprzedazy, Kwota), `dim_Placowki` (ID_Placowki, Miasto).

To jest inny świat funkcji okienkowych niż DAX (`RANK`, `WINDOW`, `OFFSET` z Twoich notatek o DAX) — SQL Server ma **własną, znacznie starszą i bardziej ugruntowaną** rodzinę funkcji okienkowych, opartą o klauzulę `OVER`. Warto nie mylić tych dwóch światów, mimo podobnych nazw funkcji.

## 1. Anatomia klauzuli `OVER` — trzy niezależne elementy

```sql
<funkcja> OVER (
    [PARTITION BY <kolumna(y)>]
    [ORDER BY <kolumna(y)>]
    [<specyfikacja_ramki>]
)
```

Wszystkie trzy elementy są **opcjonalne niezależnie od siebie**, ale ich obecność/brak zmienia znaczenie zapytania:

- **`PARTITION BY`** — dzieli wiersze na niezależne grupy (jak `GROUP BY`, ale **bez** redukowania liczby wierszy wynikowych — każdy wiersz zostaje, tylko funkcja liczy się osobno w obrębie każdej grupy). Brak `PARTITION BY` = cała tabela to jedna partycja.
- **`ORDER BY`** — definiuje kolejność, w jakiej funkcja "widzi" wiersze w obrębie partycji. Wymagane dla funkcji rankingowych (`ROW_NUMBER`, `RANK`) i offsetowych (`LAG`/`LEAD`) — bez sensownej kolejności te funkcje nie mają znaczenia. Dla funkcji agregujących (`SUM`, `AVG`) obecność `ORDER BY` **zmienia domyślną ramkę** (patrz sekcja 3) — to częste źródło zaskoczenia.
- **Specyfikacja ramki (`ROWS`/`RANGE BETWEEN ... AND ...`)** — dokładny zakres wierszy w obrębie partycji, na których funkcja operuje względem bieżącego wiersza. Sekcje 2-4 poniżej.

**Kluczowa różnica względem `GROUP BY`:** `GROUP BY` zwraca **jeden wiersz na grupę**. Funkcja okienkowa z `PARTITION BY` zwraca **jeden wynik funkcji na każdy oryginalny wiersz**, tyle że policzony w kontekście jego grupy — liczba wierszy w wyniku się nie zmienia. To pozwala pokazać jednocześnie szczegół (pojedynczą transakcję) i agregat (np. sumę całej grupy) w tym samym wierszu.

## 2. `ROWS` vs `RANGE` — fizyczna pozycja vs wartość sortowania

To jest rozróżnienie, które **prawie zawsze** jest pomijane w podstawowych tutorialach, a ma realne znaczenie przy remisach w `ORDER BY`.

**`ROWS BETWEEN ... AND ...`** — liczy **fizyczne wiersze** względem bieżącej pozycji, niezależnie od tego, czy sąsiednie wiersze mają tę samą wartość sortowania, czy nie. "2 wiersze wcześniej" to zawsze dokładnie 2 wiersze, punkt.

**`RANGE BETWEEN ... AND ...`** — liczy względem **wartości** kolumny z `ORDER BY`, nie fizycznej pozycji. Wszystkie wiersze z tą samą wartością sortowania co bieżący wiersz są traktowane jako "ten sam punkt" — przy `RANGE`, jeśli trzy wiersze mają identyczną datę, wszystkie trzy "widzą" się nawzajem jako będące w tym samym miejscu ramki, niezależnie od fizycznej kolejności między nimi.

### Przykład ujawniający różnicę — suma skumulowana przy remisach w dacie

```sql
SELECT
    DataSprzedazy, Kwota,
    SUM(Kwota) OVER (ORDER BY DataSprzedazy ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS SumaROWS,
    SUM(Kwota) OVER (ORDER BY DataSprzedazy RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS SumaRANGE
FROM fact_Sprzedaz
WHERE ID_Klienta = 'K001'
ORDER BY DataSprzedazy;
```

Jeśli klient K001 ma **dwie transakcje tego samego dnia** (np. 15.03: 100 zł i 15.03: 50 zł):

- **`SumaROWS`** — liczy wiersz po wierszu w fizycznej kolejności zwróconej przez silnik (która dla remisów **nie jest gwarantowana** bez dodatkowej kolumny w `ORDER BY` — patrz sekcja 6): pierwsza transakcja 15.03 dostanie skumulowaną sumę "do niej", druga transakcja 15.03 dostanie sumę "do niej" (już z uwzględnieniem pierwszej).
- **`SumaRANGE`** — **obie** transakcje z 15.03 dostaną **tę samą wartość** skumulowanej sumy (uwzględniającą obie transakcje z tego dnia naraz), bo `RANGE` traktuje remisy jako jeden punkt, nie rozróżnia ich kolejności.

**Konsekwencja praktyczna:** jeśli Twoim celem jest klasyczna "suma narastająco, wiersz po wierszu" (typowy running total w raporcie transakcyjnym) — **zawsze pisz jawnie `ROWS`, nigdy nie polegaj na domyślnym zachowaniu bez sprawdzenia**, które (sekcja 3) bywa `RANGE`, nie `ROWS`.

## 3. Domyślna ramka — pułapka, którą trzeba znać na pamięć

**Gdy `ORDER BY` jest obecny, a specyfikacja ramki jest pominięta, SQL Server domyślnie stosuje `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`** — nie `ROWS`. To jest zaskakujące dla większości osób piszących funkcje okienkowe po raz pierwszy, bo w praktyce najczęściej chce się `ROWS`.

```sql
-- Te dwa zapisy są RÓWNOWAŻNE (domyślna ramka to RANGE, nie ROWS):
SUM(Kwota) OVER (ORDER BY DataSprzedazy)
SUM(Kwota) OVER (ORDER BY DataSprzedazy RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
```

Przy braku remisów w kolumnie `ORDER BY` różnica między domyślnym `RANGE` a jawnym `ROWS` jest **niewidoczna w wyniku** — dlatego wielu ludzi pisze kod bez świadomości tej pułapki przez lata, dopóki nie trafi na dane z remisami (jak dwie transakcje tego samego dnia z sekcji 2), które nagle dają "dziwny" wynik.

**Reguła praktyczna: zawsze pisz ramkę jawnie (`ROWS BETWEEN ...`), nawet gdy chcesz zachowania identycznego z domyślnym.** To eliminuje całą klasę błędów związanych z niejednoznacznością przy przyszłych zmianach w danych (np. gdy dane, które kiedyś nie miały remisów w dacie, zaczną je mieć).

## 4. Pełny słownik specyfikacji ramki

```sql
[ROWS | RANGE] BETWEEN <początek> AND <koniec>
```

| Wartość | Znaczenie |
|---|---|
| `UNBOUNDED PRECEDING` | Od pierwszego wiersza partycji |
| `N PRECEDING` | N wierszy/wartości przed bieżącym (np. `3 PRECEDING`) |
| `CURRENT ROW` | Bieżący wiersz |
| `N FOLLOWING` | N wierszy/wartości po bieżącym |
| `UNBOUNDED FOLLOWING` | Do ostatniego wiersza partycji |

**Typowe, nazwane kombinacje, warte zapamiętania:**

| Kombinacja | Efekt | Typowe zastosowanie |
|---|---|---|
| `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` | Suma/wartość narastająco od początku do bieżącego wiersza | Running total |
| `ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING` | Od bieżącego wiersza do końca partycji | "Pozostało do końca" (np. ile sprzedaży jeszcze nastąpi) |
| `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` | Cała partycja, dla każdego wiersza ten sam wynik | Suma całkowita grupy widoczna przy każdym wierszu (np. do liczenia % udziału) |
| `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` | Bieżący + 2 poprzednie wiersze (3 wiersze łącznie) | Średnia/suma krocząca "3-okresowa" |
| `ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING` | Bieżący + 1 przed + 1 po (3 wiersze wyśrodkowane) | Wygładzanie symetryczne |
| *(pominięta ramka, brak `ORDER BY`)* | Cała partycja | Odpowiednik `UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`, bez sensu kolejności |

### Przykład — suma całkowita grupy widoczna przy każdym wierszu (do liczenia % udziału)

```sql
SELECT
    s.ID_Klienta, s.DataSprzedazy, s.Kwota,
    SUM(s.Kwota) OVER (
        PARTITION BY s.ID_Klienta
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS SumaCalkowitaKlienta,
    CAST(s.Kwota AS DECIMAL(18,4)) / SUM(s.Kwota) OVER (PARTITION BY s.ID_Klienta) AS UdzialProcentowy
FROM fact_Sprzedaz s;
```

**Uwaga:** `SUM(s.Kwota) OVER (PARTITION BY s.ID_Klienta)` **bez `ORDER BY`** domyślnie stosuje ramkę całej partycji (`UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`) — to jest jedyny przypadek, gdzie pominięcie ramki jest w pełni bezpieczne i przewidywalne, bo bez `ORDER BY` pojęcie "poprzedni"/"następny" wiersz w ogóle nie ma znaczenia, więc silnik i tak liczy całą partycję.

### Przykład — średnia krocząca 3-okresowa (analogia do `WINDOW`/`DATESINPERIOD` z Twoich notatek DAX)

```sql
SELECT
    DataSprzedazy, Kwota,
    AVG(Kwota) OVER (
        ORDER BY DataSprzedazy
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS SredniaKroczaca3Okresy
FROM fact_Sprzedaz
WHERE ID_Placowki = 1;
```

Bieżący wiersz + 2 poprzednie = 3 wiersze łącznie uśredniane — to jest dokładnie ten sam koncept co `WINDOW(-2, REL, 0, REL, ...)` w DAX, tylko w innej składni SQL.

## 5. Funkcje rankingowe — `ROW_NUMBER`, `RANK`, `DENSE_RANK`, `NTILE`

Wszystkie cztery wymagają `ORDER BY` w `OVER(...)` — bez kolejności ranking nie ma sensu.

```sql
SELECT
    k.ID_Klienta, k.Nazwa, SumaSprzedazy,
    ROW_NUMBER() OVER (ORDER BY SumaSprzedazy DESC) AS NumerWiersza,
    RANK()       OVER (ORDER BY SumaSprzedazy DESC) AS Ranga,
    DENSE_RANK() OVER (ORDER BY SumaSprzedazy DESC) AS RangaGesta,
    NTILE(10)    OVER (ORDER BY SumaSprzedazy DESC) AS Decyl
FROM (
    SELECT s.ID_Klienta, SUM(s.Kwota) AS SumaSprzedazy
    FROM fact_Sprzedaz s GROUP BY s.ID_Klienta
) AS Agregat
JOIN dim_Klienci k ON k.ID_Klienta = Agregat.ID_Klienta;
```

**Różnice przy remisach (dwaj klienci z identyczną `SumaSprzedazy`):**

| Funkcja | Zachowanie przy remisie | Przykład (2 remisujące na 2. miejscu) |
|---|---|---|
| `ROW_NUMBER()` | Zawsze unikalny numer, remis rozstrzygany arbitralnie (fizyczną kolejnością, chyba że `ORDER BY` jednoznacznie różnicuje) | 1, 2, 3, 4 |
| `RANK()` | Remisujące dostają tę samą rangę, kolejna wartość "przeskakuje" (jak `SKIP` w DAX `RANKX`) | 1, 2, 2, 4 |
| `DENSE_RANK()` | Remisujące dostają tę samą rangę, kolejna wartość **nie przeskakuje** (jak `DENSE` w DAX) | 1, 2, 2, 3 |
| `NTILE(N)` | Dzieli na N w miarę równych grup (np. decyle) — **remisy mogą trafić do różnych grup**, bo `NTILE` dba o równoliczność grup, nie o spójność wartości | Zależy od pozycji fizycznej |

**`NTILE(10)`** to bezpośredni, wbudowany odpowiednik "podziału na decyle" — znacznie prostszy niż ręczne liczenie `CEILING(RANK/COUNT*10, 1)`, które budowaliśmy w DAX. Jeśli kiedyś potrzebujesz decyli w warstwie SQL (np. przed załadowaniem do modelu Power BI), `NTILE` to naturalny, gotowy wybór — **ale ma inną semantykę przy remisach niż ręczny wzorzec z DAX** (dba o równoliczność grup, nie o spójne przypisanie identycznych wartości do tej samej grupy) — warto to świadomie porównać, jeśli oba mechanizmy mają dawać spójne wyniki w różnych warstwach systemu.

**`ROW_NUMBER()` z niejednoznacznym `ORDER BY` — niedeterminizm:** jeśli sortujesz tylko po `SumaSprzedazy DESC`, a są remisy, **SQL Server nie gwarantuje**, który z remisujących dostanie numer 2, a który 3 — wynik może się różnić między uruchomieniami (zależnie od planu wykonania). Jeśli potrzebujesz w pełni deterministycznego, powtarzalnego wyniku, dodaj kolumnę tie-breaker: `ORDER BY SumaSprzedazy DESC, ID_Klienta ASC`.

## 6. Funkcje offsetowe — `LAG`, `LEAD`, `FIRST_VALUE`, `LAST_VALUE`

### `LAG` / `LEAD` — wartość z poprzedniego/następnego wiersza

```sql
LAG  ( <wyrażenie> [, <przesunięcie>] [, <wartość_domyślna>] ) OVER ( [PARTITION BY ...] ORDER BY ... )
LEAD ( <wyrażenie> [, <przesunięcie>] [, <wartość_domyślna>] ) OVER ( [PARTITION BY ...] ORDER BY ... )
```

- `<przesunięcie>` — domyślnie `1` (jeden wiersz wstecz/wprzód), można podać dowolną liczbę.
- `<wartość_domyślna>` — co zwrócić, gdy nie ma wiersza o żądanym przesunięciu (np. pierwszy wiersz partycji nie ma poprzednika dla `LAG`) — domyślnie `NULL`, jeśli nie podasz inaczej.

**Przykład — poprzednia transakcja tego samego klienta (dokładny odpowiednik `OFFSET(-1, ...)` z DAX):**

```sql
SELECT
    s.ID_Klienta, s.DataSprzedazy, s.Kwota,
    LAG(s.Kwota, 1, 0) OVER (PARTITION BY s.ID_Klienta ORDER BY s.DataSprzedazy) AS PoprzedniaKwota,
    s.Kwota - LAG(s.Kwota, 1, 0) OVER (PARTITION BY s.ID_Klienta ORDER BY s.DataSprzedazy) AS Zmiana
FROM fact_Sprzedaz s;
```

`LAG(s.Kwota, 1, 0)` — jeśli to pierwsza transakcja klienta (brak poprzednika), zwróci `0` (jawna wartość domyślna) zamiast `NULL` — przydatne, żeby `Zmiana` nie stała się `NULL` dla pierwszej transakcji każdego klienta.

**`LAG`/`LEAD` nie akceptują specyfikacji ramki `ROWS`/`RANGE`** — działają zawsze względem fizycznej kolejności `ORDER BY` w obrębie partycji, niezależnie od ramki (bo pytają o konkretny, pojedynczy wiersz przesunięty o N pozycji, nie o zakres).

### `FIRST_VALUE` / `LAST_VALUE` — wartość z pierwszego/ostatniego wiersza ramki

```sql
FIRST_VALUE ( <wyrażenie> ) OVER ( [PARTITION BY ...] ORDER BY ... [<ramka>] )
LAST_VALUE  ( <wyrażenie> ) OVER ( [PARTITION BY ...] ORDER BY ... [<ramka>] )
```

**Najczęstsza pułapka w całym T-SQL: `LAST_VALUE` z domyślną ramką prawie nigdy nie robi tego, czego oczekujesz.**

```sql
-- BŁĘDNE (typowy, częsty błąd) — LAST_VALUE z pominiętą ramką
SELECT
    DataSprzedazy, Kwota,
    LAST_VALUE(Kwota) OVER (ORDER BY DataSprzedazy) AS OstatniaKwota   -- NIE działa jak oczekujesz!
FROM fact_Sprzedaz
WHERE ID_Klienta = 'K001';
```

Pamiętasz z sekcji 3: domyślna ramka to `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` — czyli **"od początku do bieżącego wiersza"**. `LAST_VALUE` w tej ramce zwraca więc **wartość bieżącego wiersza**, nie ostatniego wiersza całej partycji! Dla każdego wiersza `OstatniaKwota` będzie równa jego własnej `Kwota`, co jest bezużyteczne i mylące.

**Poprawna wersja — jawna ramka obejmująca całą partycję:**

```sql
SELECT
    DataSprzedazy, Kwota,
    LAST_VALUE(Kwota) OVER (
        ORDER BY DataSprzedazy
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS OstatniaKwota
FROM fact_Sprzedaz
WHERE ID_Klienta = 'K001';
```

Teraz ramka jawnie obejmuje **całą** partycję (od początku do końca), więc `LAST_VALUE` faktycznie zwraca wartość z ostatniego wiersza w kolejności `DataSprzedazy`, dla każdego wiersza jednakowo.

**`FIRST_VALUE` nie ma tego samego problemu w praktyce** — przy domyślnej ramce (`... AND CURRENT ROW`) pierwszy wiersz partycji zawsze jest w zasięgu, więc `FIRST_VALUE` zwraca poprawną wartość nawet bez jawnej ramki. Mimo to **zalecam jawną ramkę zawsze przy obu funkcjach**, dla spójności i czytelności kodu, a nie tylko tam, gdzie akurat jest to konieczne.

## 7. `PERCENT_RANK` i `CUME_DIST` — pozycja względna w rozkładzie

Rzadziej używane, ale warte znajomości — dają wynik jako **ułamek (0-1)**, nie liczbę porządkową.

```sql
SELECT
    k.ID_Klienta, SumaSprzedazy,
    PERCENT_RANK() OVER (ORDER BY SumaSprzedazy) AS PozycjaProcentowa,
    CUME_DIST()    OVER (ORDER BY SumaSprzedazy) AS DystrybuantaSkumulowana
FROM ( ... ) AS Agregat;
```

- **`PERCENT_RANK()`** = `(RANK() - 1) / (liczba_wierszy - 1)` — 0 dla pierwszego wiersza (w kolejności `ORDER BY`), 1 dla ostatniego. Przydatne do "gdzie jestem w skali 0-100%" bez samodzielnego liczenia formuły.
- **`CUME_DIST()`** (cumulative distribution) = odsetek wierszy o wartości **mniejszej lub równej** bieżącej — odpowiedź na "jaki % populacji ma wynik nie lepszy ode mnie". Różni się od `PERCENT_RANK` przy remisach i w punktach granicznych (nigdy nie jest dokładnie 0, zawsze >0).

**Praktyczne zastosowanie:** gdybyś chciał w warstwie SQL (przed Power BI) obliczyć coś analogicznego do Twojej miary decylowej z DAX, ale jako ciągły percentyl zamiast dyskretnych 10 grup — `PERCENT_RANK`/`CUME_DIST` to naturalny, gotowy wybór, tańszy niż ręczne liczenie.

## 8. Funkcje agregujące jako funkcje okienkowe — `COUNT`, `MIN`, `MAX` z `OVER`

Każda standardowa funkcja agregująca (`SUM`, `AVG`, `COUNT`, `MIN`, `MAX`, `STDEV`, `VAR`) może wystąpić z `OVER(...)` — działa dokładnie tak samo jak `SUM` w przykładach wyżej, tylko z inną agregacją.

**Przykład — liczba transakcji klienta narastająco + minimalna/maksymalna kwota w historii do tego punktu:**

```sql
SELECT
    s.ID_Klienta, s.DataSprzedazy, s.Kwota,
    COUNT(*) OVER (
        PARTITION BY s.ID_Klienta ORDER BY s.DataSprzedazy
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS LiczbaTransakcjiNarastajaco,
    MAX(s.Kwota) OVER (
        PARTITION BY s.ID_Klienta ORDER BY s.DataSprzedazy
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS NajwiekszaKwotaDotad
FROM fact_Sprzedaz s;
```

`NajwiekszaKwotaDotad` — dla każdej transakcji pokazuje największą kwotę **spośród tej i wszystkich wcześniejszych** transakcji tego klienta — klasyczny wzorzec "rekord życiowy do tego momentu", częsty w analizach lojalnościowych/sprzedażowych.

**`COUNT(*)` vs `COUNT(kolumna)` w oknie** — ta sama zasada co poza oknem: `COUNT(*)` liczy wszystkie wiersze ramki, `COUNT(kolumna)` pomija `NULL`-e w tej konkretnej kolumnie. Warto świadomie wybrać, którego potrzebujesz.

## 9. Połączenie kilku funkcji okienkowych w jednym zapytaniu — pełny przykład

To podsumowujący przykład łączący większość powyższych elementów w jednym, realistycznym zapytaniu analitycznym:

```sql
SELECT
    s.ID_Klienta,
    s.DataSprzedazy,
    s.Kwota,
    -- Running total (jawnie ROWS, zgodnie z sekcją 3)
    SUM(s.Kwota) OVER (
        PARTITION BY s.ID_Klienta ORDER BY s.DataSprzedazy
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS SumaNarastajaco,
    -- Średnia krocząca 3-transakcyjna
    AVG(s.Kwota) OVER (
        PARTITION BY s.ID_Klienta ORDER BY s.DataSprzedazy
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS SredniaKroczaca3,
    -- Poprzednia transakcja
    LAG(s.Kwota) OVER (PARTITION BY s.ID_Klienta ORDER BY s.DataSprzedazy) AS PoprzedniaKwota,
    -- Ranking transakcji klienta wg kwoty
    RANK() OVER (PARTITION BY s.ID_Klienta ORDER BY s.Kwota DESC) AS RangaKwotyKlienta,
    -- Udział % transakcji w całkowitej sprzedaży klienta
    CAST(s.Kwota AS DECIMAL(18,4)) / SUM(s.Kwota) OVER (PARTITION BY s.ID_Klienta) AS UdzialProcentowyKlienta
FROM fact_Sprzedaz s
ORDER BY s.ID_Klienta, s.DataSprzedazy;
```

Zwróć uwagę: każda funkcja okienkowa w tym zapytaniu ma **własną, niezależną** klauzulę `OVER(...)` — mogą mieć różne `PARTITION BY`, różne `ORDER BY`, różne ramki, nawet w obrębie tego samego `SELECT`. To jest normalne i częste — nie musisz (i zwykle nie powinieneś) sztucznie ujednolicać wszystkich okien w zapytaniu do jednej specyfikacji.

## 10. Podsumowanie — ściąga

| Potrzebujesz | Funkcja / wzorzec |
|---|---|
| Unikalny numer wiersza w partycji | `ROW_NUMBER()` |
| Ranking z "dziurami" po remisie | `RANK()` |
| Ranking bez "dziur" po remisie | `DENSE_RANK()` |
| Podział na N w miarę równych grup (np. decyle) | `NTILE(N)` |
| Wartość z poprzedniego/następnego wiersza | `LAG()` / `LEAD()` |
| Wartość z pierwszego/ostatniego wiersza ramki | `FIRST_VALUE()` / `LAST_VALUE()` — **zawsze z jawną ramką** |
| Pozycja względna w rozkładzie (0-1) | `PERCENT_RANK()` / `CUME_DIST()` |
| Suma/średnia narastająco od początku | `SUM()`/`AVG() OVER (... ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)` |
| Suma/średnia krocząca N-okresowa | `... ROWS BETWEEN (N-1) PRECEDING AND CURRENT ROW` |
| Suma/wartość całej grupy przy każdym wierszu (np. do % udziału) | `... OVER (PARTITION BY ...)` bez `ORDER BY`, albo z jawną ramką `UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` |
| Remisy w dacie/wartości sortowania mają być traktowane jako jeden punkt | `RANGE` zamiast `ROWS` (rzadziej potrzebne — domyślnie i tak `RANGE`, patrz sekcja 3!) |
| Pewność co do zachowania przy remisach, deterministyczny wynik | Zawsze dopisz tie-breaker do `ORDER BY` (np. klucz unikalny) |

**Trzy zasady do zapamiętania na zawsze:**
1. **Zawsze pisz ramkę jawnie** (`ROWS BETWEEN ...`), nie polegaj na domyślnej `RANGE ... CURRENT ROW`.
2. **`LAST_VALUE` bez jawnej ramki obejmującej całą partycję prawie zawsze zwraca zły wynik** — to jest błąd numer jeden w praktyce z tej całej rodziny funkcji.
3. **Dodawaj tie-breaker do `ORDER BY`** w funkcjach rankingowych, jeśli zależy Ci na deterministycznym, powtarzalnym wyniku przy remisach.